# Event displays for jets with unexpected flavour content

Standalone RDataFrame version of the parton-level IFN clustering from `jet-flv-test-histmaker.py`
(hadronisation partons → flavoured pseudojets → exclusive 2-jet `ee_kt` + IFN).
Events where the jet flavour content does not match the expectation for the sample are drawn as a
2D event display in the plane spanned by the two jets:

* x axis: direction of jet 1, y axis: component of jet 2 orthogonal to jet 1.
* Partons are arrows (projected momentum), coloured by the jet they are clustered into and labelled with their PDG ID.
  Dashed arrows point mostly out of the plane (|p_⊥ plane| > |p_in plane|), so their drawn length is misleading.
* Expectation: for H → qq̄ one jet with q content +1 and one with −1, all other flavours 0; for H → gg all contents 0.

Plots go to `outputs/jet-test/event_displays/parton_ifn/<sample>/` (one PDF per event plus one combined PDF).

In [1]:
import os
import glob
from collections import Counter

import numpy as np
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)
# Loading libFCCAnalyses prints ~300 kB of "Info in <TCling::LoadPCM>" messages, which
# fills JupyROOT's output pipe and hangs the kernel. Only show warnings and above.
ROOT.gErrorIgnoreLevel = ROOT.kWarning
# Load libFastJet first, otherwise libFCCAnalyses resolves it to the central
# cvmfs copy, which lacks the FlavInfo symbols needed by the IFN clustering.
ROOT.gSystem.Load("libFastJet")
ROOT.gSystem.Load("libFCCAnalyses")
ROOT.dummyLoader()

Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/lib/libROOTVecOps_rdict.pcm file does not exist
Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/lib/libMatrix_rdict.pcm file does not exist
Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/lib/libHist_rdict.pcm file does not exist
Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/lib/libGraf_rdict.pcm file does not exist
Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/lib/libGpad_rdict.pcm file does not exist
Error in <TCling::LoadPCM>: ROOT PCM /cvmfs/sft-nightlies.cern.ch/lcg/nightlies/devkey-head/Thu/ROOT/6.40.04/x86_64-el9-gcc16-opt/li

True

In [2]:
SAMPLE = "wzp6_ee_nunuH_Hbb_ecm240"
INPUT_DIR = f"/eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/{SAMPLE}"
N_FILES = 1        # number of input files to chain
N_EVENTS = 20000   # events to scan, -1 for all events in the chained files
N_DISPLAY = 10     # number of mismatched events to draw

PLOT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test", "event_displays", "parton_ifn", SAMPLE)
os.makedirs(PLOT_DIR, exist_ok=True)

FLAVOURS = ["u", "d", "s", "c", "b"]


def expected_content(sample):
    """Expected sorted per-jet content of each flavour.

    H -> qqbar: one jet with q content +1, the other -1. H -> gg: no net flavour.
    """
    q = sample.split("_H")[1][0]  # e.g. "b" for wzp6_ee_nunuH_Hbb_ecm240, "g" for Hgg
    return {f: (-1, 1) if f == q else (0, 0) for f in FLAVOURS}


EXPECTED = expected_content(SAMPLE)
EXPECTED

{'u': (0, 0), 'd': (0, 0), 's': (0, 0), 'c': (0, 0), 'b': (-1, 1)}

In [3]:
files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))[:N_FILES]
chain = ROOT.TChain("events")
for f in files:
    chain.Add(f)

df = ROOT.RDataFrame(chain)
if N_EVENTS > 0:
    df = df.Range(N_EVENTS)  # Range needs single-threaded RDF, so no EnableImplicitMT here

df = (df.Define("entry", "rdfentry_")
        .Define("partons", "FCCAnalyses::MCParticle::get_hadronisation_partons(Particle, _Particle_parents.index)")
        .Define("parton_px", "FCCAnalyses::MCParticle::get_px(partons)")
        .Define("parton_py", "FCCAnalyses::MCParticle::get_py(partons)")
        .Define("parton_pz", "FCCAnalyses::MCParticle::get_pz(partons)")
        .Define("parton_m", "FCCAnalyses::MCParticle::get_mass(partons)")
        .Define("parton_pdg", "FCCAnalyses::MCParticle::get_pdg(partons)")
        .Define("pseudo_jets", "FCCAnalyses::JetClusteringUtils::set_pseudoJets_xyzm_flav(parton_px, parton_py, parton_pz, parton_m, parton_pdg)")
        .Define("clustered_jets", "JetClustering::flv_ifn_clustering_ee_kt(2, 2, 0, 0)(pseudo_jets)")
        .Define("jets", "FCCAnalyses::JetClusteringUtils::get_pseudoJets(clustered_jets)")
        .Define("jets_constituents", "FCCAnalyses::JetClusteringUtils::get_constituents(clustered_jets)")
        .Define("jets_flavour", "FCCAnalyses::JetClusteringUtils::get_flavours(jets)"))
for var in ["px", "py", "pz", "e"]:
    df = df.Define(f"jets_{var}", f"FCCAnalyses::JetClusteringUtils::get_{var}(jets)")
for f in FLAVOURS:
    df = df.Define(f"jets_{f}_content", f"FCCAnalyses::JetClusteringUtils::get_{f}_content(jets)")

matches = " && ".join(f"All(Sort(jets_{f}_content) == ROOT::RVecI{{{lo}, {hi}}})" for f, (lo, hi) in EXPECTED.items())
df_mismatch = df.Filter(f"jets.size() == 2 && !({matches})", "flavour content mismatch")

n_total = df.Count()
columns = (["entry", "parton_px", "parton_py", "parton_pz", "parton_pdg", "jets_constituents", "jets_flavour"]
           + [f"jets_{v}" for v in ["px", "py", "pz", "e"]] + [f"jets_{f}_content" for f in FLAVOURS])
events = df_mismatch.AsNumpy(columns)  # triggers the event loop
n_mismatch = len(events["entry"])
print(f"{SAMPLE}: {n_mismatch} / {n_total.GetValue()} events ({n_mismatch / n_total.GetValue():.2%}) don't match the expectation")

#--------------------------------------------------------------------------
#                         FastJet release 3.5.1
#                 M. Cacciari, G.P. Salam and G. Soyez                  
#     A software package for jet finding and analysis at colliders      
#                           https://fastjet.fr                           
#	                                                                      
# Please cite EPJC72(2012)1896 [arXiv:1111.6097] if you use this package
# for scientific work and optionally PLB641(2006)57 [hep-ph/0512210].   
#                                                                       
# FastJet is provided without warranty under the GNU GPL v2 or higher.  
# It uses T. Chan's closest pair algorithm, S. Fortune's Voronoi code
# and 3rd party plugin jet algorithms. See COPYING file for details.
#--------------------------------------------------------------------------


wzp6_ee_nunuH_Hbb_ecm240: 880 / 20000 events (4.40%) don't match the expectation


In [4]:
# What kind of mismatches are there? Jet flavours (as strings) of the mismatched events, most common first.
patterns = Counter(" | ".join(sorted(str(s).strip() for s in fl)) for fl in events["jets_flavour"])
for pattern, n in patterns.most_common(15):
    print(f"{n:6d}  ({n / n_mismatch:6.1%})  {pattern}")

   863  ( 98.1%)  [g] | [g]
     3  (  0.3%)  [s b ] | [sbar bbar ]
     3  (  0.3%)  [u bbar ] | [ubar b ]
     3  (  0.3%)  [s bbar ] | [sbar b ]
     2  (  0.2%)  [d bbar ] | [dbar b ]
     2  (  0.2%)  [b b ] | [bbar bbar ]
     2  (  0.2%)  [s ] | [sbar ]
     1  (  0.1%)  [d ] | [dbar ]
     1  (  0.1%)  [c b ] | [cbar bbar ]


In [5]:
JET_COLOURS = [ROOT.kRed + 1, ROOT.kAzure + 2]


def event_location(entry):
    """Input file and entry within it, e.g. for edm4hep2json."""
    chain.LoadTree(int(entry))
    return chain.GetFile().GetName(), int(entry) - chain.GetChainOffset()


def jet_plane(j1, j2):
    """Orthonormal basis (e1, e2, n) with e1 along jet 1 and jet 2 in the (e1, e2) plane."""
    e1 = j1 / np.linalg.norm(j1)
    v = j2 - j2.dot(e1) * e1
    if np.linalg.norm(v) < 1e-6 * np.linalg.norm(j2):  # back-to-back jets: plane not defined, pick any
        v = np.cross(e1, [0., 0., 1.] if abs(e1[2]) < 0.9 else [1., 0., 0.])
    e2 = v / np.linalg.norm(v)
    return e1, e2, np.cross(e1, e2)


def content_string(i_ev, i_jet):
    return " ".join(f"{f}{int(events[f'jets_{f}_content'][i_ev][i_jet]):+d}" for f in FLAVOURS)


def event_display(i_ev):
    entry = int(events["entry"][i_ev])
    jets = [np.array([events[f"jets_{v}"][i_ev][i] for v in ["px", "py", "pz"]]) for i in range(2)]
    partons = np.array([np.asarray(events[f"parton_{v}"][i_ev]) for v in ["px", "py", "pz"]]).T
    pdgs = np.asarray(events["parton_pdg"][i_ev]).astype(int)
    jet_of = np.full(len(pdgs), -1)
    for i_jet, constituents in enumerate(events["jets_constituents"][i_ev]):
        jet_of[np.asarray(constituents)] = i_jet

    e1, e2, n = jet_plane(*jets)
    proj = lambda p: (p @ e1, p @ e2)
    px, py = proj(partons)
    out = np.abs(partons @ n)
    jx, jy = zip(*(proj(j) for j in jets))

    r = 1.15 * max(np.max(np.abs(np.concatenate([px, py, jx, jy]))), 1.)
    c = ROOT.TCanvas(f"evt_{SAMPLE}_{entry}", "", 800, 800)
    c.SetMargin(0.12, 0.05, 0.1, 0.1)
    frame = c.DrawFrame(-r, -r, r, r)
    frame.SetTitle(f"{SAMPLE}, entry {entry};p #upoint #hat{{e}}_{{1}} (jet 1) [GeV];p #upoint #hat{{e}}_{{2}} [GeV]")
    keep = [frame]

    for i_jet in range(2):  # jet axes: thick, translucent
        a = ROOT.TArrow(0., 0., jx[i_jet], jy[i_jet], 0.025, "|>")
        a.SetLineWidth(8)
        a.SetLineColorAlpha(JET_COLOURS[i_jet], 0.3)
        a.SetFillColorAlpha(JET_COLOURS[i_jet], 0.3)
        a.Draw()
        keep.append(a)

    for x, y, o, pdg, i_jet in zip(px, py, out, pdgs, jet_of):
        colour = JET_COLOURS[i_jet] if i_jet >= 0 else ROOT.kGray + 2
        a = ROOT.TArrow(0., 0., x, y, 0.012, "|>")
        a.SetLineWidth(2)
        a.SetLineColor(colour)
        a.SetFillColor(colour)
        if o > np.hypot(x, y):
            a.SetLineStyle(2)
        a.Draw()
        is_quark = abs(pdg) <= 6
        label = ROOT.TLatex(x * 1.04, y * 1.04, str(pdg))
        label.SetTextAlign((31 if x < 0 else 11) + (0 if y < 0 else 2) - 1)  # place label away from the origin
        label.SetTextSize(0.032 if is_quark else 0.022)
        label.SetTextFont(62 if is_quark else 42)
        label.SetTextColor(colour)
        label.Draw()
        keep += [a, label]

    leg = ROOT.TLegend(0.14, 0.12, 0.6, 0.27)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize(0.025)
    for i_jet in range(2):
        line = ROOT.TLine()
        line.SetLineColor(JET_COLOURS[i_jet])
        line.SetLineWidth(3)
        leg.AddEntry(line, f"jet {i_jet + 1}: {content_string(i_ev, i_jet)}", "l")
        keep.append(line)
    dashed = ROOT.TLine()
    dashed.SetLineStyle(2)
    leg.AddEntry(dashed, "mostly out of the jet plane", "l")
    leg.Draw()
    keep += [leg, dashed]

    c._keep = keep  # keep the primitives alive
    return c

In [6]:
canvases = []
for i_ev in range(min(N_DISPLAY, n_mismatch)):
    entry = int(events["entry"][i_ev])
    fname, local_entry = event_location(entry)
    print(f"entry {entry}: {[str(s).strip() for s in events['jets_flavour'][i_ev]]}  "
          f"->  edm4hep2json -e {local_entry} -l Particle -o out.json {fname}")
    c = event_display(i_ev)
    c.SaveAs(os.path.join(PLOT_DIR, f"entry_{entry}.pdf"))
    c.Draw()
    canvases.append(c)

if canvases:
    combined = os.path.join(PLOT_DIR, f"all_events_{SAMPLE}.pdf")
    canvases[0].Print(combined + "[")
    for c in canvases:
        c.Print(combined, f"Title:{c.GetName()}")
    canvases[-1].Print(combined + "]")

entry 35: ['[g]', '[g]']  ->  edm4hep2json -e 35 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 44: ['[g]', '[g]']  ->  edm4hep2json -e 44 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 60: ['[g]', '[g]']  ->  edm4hep2json -e 60 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 114: ['[g]', '[g]']  ->  edm4hep2json -e 114 -l Particle -o out.json root://eospublic.cern.ch//eos/experiment/fcc/ee/generation/DelphesEvents/pre_summer2026_training/IDEA/wzp6_ee_nunuH_Hbb_ecm240/events_505359172.root
entry 196: ['[g]', '[g]']  ->  edm4hep2json -e 196 -l Particle -o out.json root://eospublic.cern.ch//eos/exper